# Install Java


In [ ]:
#!apt-get install openjdk-8-jdk
!apt-get install default-jre

In [2]:
!java -version # This command displays Java version.

openjdk version "11.0.22" 2024-01-16
OpenJDK Runtime Environment (build 11.0.22+7-post-Ubuntu-0ubuntu222.04.1)
OpenJDK 64-Bit Server VM (build 11.0.22+7-post-Ubuntu-0ubuntu222.04.1, mixed mode, sharing)


# Install H2O

In [ ]:
!pip install H2O

In [4]:
import h2o
h2o.init()

Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
  Java Version: openjdk version "11.0.22" 2024-01-16; OpenJDK Runtime Environment (build 11.0.22+7-post-Ubuntu-0ubuntu222.04.1); OpenJDK 64-Bit Server VM (build 11.0.22+7-post-Ubuntu-0ubuntu222.04.1, mixed mode, sharing)
  Starting server from /usr/local/lib/python3.10/dist-packages/h2o/backend/bin/h2o.jar
  Ice root: /tmp/tmph4wpp8bv
  JVM stdout: /tmp/tmph4wpp8bv/h2o_unknownUser_started_from_python.out
  JVM stderr: /tmp/tmph4wpp8bv/h2o_unknownUser_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,03 secs
H2O_cluster_timezone:,Etc/UTC
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.2
H2O_cluster_version_age:,20 days
H2O_cluster_name:,H2O_from_python_unknownUser_v7g90w
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,3.170 Gb
H2O_cluster_total_cores:,2
H2O_cluster_allowed_cores:,2
H2O_cluster_status:,"locked, healthy"


In [5]:
h2o.cluster().show_status() # Check the status and health of the H2O cluster

H2O_cluster_uptime:,1 min 17 secs
H2O_cluster_timezone:,Etc/UTC
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.2
H2O_cluster_version_age:,20 days
H2O_cluster_name:,H2O_from_python_unknownUser_v7g90w
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,3.170 Gb
H2O_cluster_total_cores:,2
H2O_cluster_allowed_cores:,2
H2O_cluster_status:,"locked, healthy"


# Tunel to localhost

In [6]:
!npm install -g localtunnel

/tools/node/bin/lt -> /tools/node/lib/node_modules/localtunnel/bin/lt.js
+ localtunnel@2.0.2
added 22 packages from 22 contributors in 2.261s


In [7]:
get_ipython().system_raw('lt --port 54321 >> url.txt &')

In [8]:
!cat url.txt

your url is: https://violet-beds-make.loca.lt


In [9]:
!curl https://loca.lt/mytunnelpassword

34.86.55.107

# Data Preparation and simple statistics
Although it may seem like you are manipulating the data in Python, once the
data has been passed to H2O, all data munging occurs in the H2O
instance. You are limited by the total amount of memory allocated to the H2O
instance, not by Python's ability to handle data.

In [ ]:
airlinesURL = 'https://s3.amazonaws.com/h2o-airlines-unpacked/allyears2k.csv' # File path
# The U.S. Department of Transportation's (DOT) Bureau of Transportation Statistics (BTS)
# tracks the on-time performance of domestic flights operated by large air carriers.

airlines_hex = h2o.import_file(airlinesURL, destination_frame = 'airlines_hex')
airlines_hex.head()

In [ ]:
# Summary statistics
airlines_hex.describe()

In [12]:
# Quantiles of the ArrDelay column
quantiles = airlines_hex['ArrDelay'].quantile(prob = [0.25, 0.5, 0.75, 1.0])
print(quantiles)

  Probs    ArrDelayQuantiles
   0.25                   -6
   0.5                     2
   0.75                   14
   1                     475
[4 rows x 2 columns]



In [13]:
# Histogram of the ArrDelay column
airlines_hex['ArrDelay'].hist()

breaks,counts,mids_true,mids,widths
-31.3529,nan,nan,nan,nan
0.294118,173,-31.5,-15.5294,31.6471
31.9412,19364,-15.5,16.1176,31.6471
63.5882,18089,0.5,47.7647,31.6471
95.2353,3163,16,79.4118,31.6471
126.882,1067,32,111.059,31.6471
158.529,444,48,142.706,31.6471
190.176,221,63.5,174.353,31.6471
221.824,116,79.5,206,31.6471
253.471,60,95.5,237.647,31.6471


In [14]:
# Check if any column is a factor
any(airlines_hex.type(col) == 'enum' for col in airlines_hex.names)

True

In [15]:
# Group by 'Origin' and calculate the number of rows, removing NA values
airlines_hex.group_by('Origin').count(na='rm').get_frame()

Origin,nrow
ABE,59
ABQ,876
ACY,31
ALB,75
AMA,11
ANC,1
ATL,754
AUS,835
AVP,154
BDL,141


In [16]:
# Group by 'Month' and calculate the number of rows, removing NA values
airlines_hex.group_by('Month').count(na='rm').get_frame()

Month,nrow
1,41979
10,1999


In [17]:
# Group by 'Month' and calculate the number of cancellations, removing NA values
cancellations_by_month = airlines_hex.group_by('Month').sum('Cancelled', na='rm').get_frame()
print(cancellations_by_month)

# Group by 'Month' and calculate the number of flights, removing NA values
flights_by_month = airlines_hex.group_by('Month').count(na='rm').get_frame()
print(flights_by_month)

# Calculate cancellation rate
cancellation_rate = cancellations_by_month['sum_Cancelled'] / flights_by_month['nrow']
print(cancellation_rate)

# Combine month and cancellation rate into a single table
rates_table = flights_by_month.cbind(cancellation_rate)
rates_table.set_name(2, 'Cancellation_Rate')
print(rates_table)

  Month    sum_Cancelled
      1             1067
     10               19
[2 rows x 2 columns]

  Month    nrow
      1   41979
     10    1999
[2 rows x 2 columns]

  sum_Cancelled
     0.0254175
     0.00950475
[2 rows x 1 column]

  Month    nrow    Cancellation_Rate
      1   41979           0.0254175
     10    1999           0.00950475
[2 rows x 3 columns]



In [18]:
# Generate a frequency table for the 'Cancelled' column
airlines_hex['Cancelled'].table()

# Because H2O can handle larger datasets, it is possible to generate tables that are larger than Python's capacity, so use caution when executing this command.

Cancelled,Count
0,42892
1,1086


# Data Manipulation

In [19]:
from sklearn.datasets import load_iris
import pandas as pd

# Load iris dataset from sklearn and convert to pandas DataFrame
iris = load_iris()
iris_df = pd.DataFrame(data = iris.data, columns = iris.feature_names)
iris_df['target'] = iris.target

# Convert pandas DataFrame to h2o Frame
iris_hex = h2o.H2OFrame(iris_df, destination_frame = 'iris_hex')
# Ensure the target column is treated as a categorical factor
iris_hex['target'] = iris_hex['target'].asfactor()

# List all objects in h2o
h2o.ls()

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


/usr/local/lib/python3.10/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install datatable (for Python 3.9 or lower), or polars and pyarrow (for Python 3.10 or above) and activate it using:

with h2o.utils.threading.local_context(polars_enabled=True, datatable_enabled=True):
    pandas_df = h2o_df.as_data_frame()

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


,key
0,airlines_hex
1,iris_hex
2,py_10_sid_a081
3,py_12_sid_a081
4,py_14_sid_a081
5,py_16_sid_a081
6,py_17_sid_a081
7,py_18_sid_a081
8,py_19_sid_a081
9,py_1_sid_a081


In [20]:
# Splitting Frames
# h2o.splitFrame() does not give an exact split. H2O is designed
# to be efficient on big data using a probabilistic splitting method rather than an
# exact split. On small datasets, the sizes of the resulting splits will deviate from
# the expected value more than on big data, where they will be very close to
# exact.
iris_split = iris_hex.split_frame(ratios=[0.75], seed = 1234)

# Creates training set from 1st data set in split
iris_train = iris_split[0]

# Creates testing set from 2nd data set in split
iris_test = iris_split[1]

# List all objects in h2o
h2o.ls()

/usr/local/lib/python3.10/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install datatable (for Python 3.9 or lower), or polars and pyarrow (for Python 3.10 or above) and activate it using:

with h2o.utils.threading.local_context(polars_enabled=True, datatable_enabled=True):
    pandas_df = h2o_df.as_data_frame()

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


,key
0,airlines_hex
1,iris_hex
2,py_10_sid_a081
3,py_12_sid_a081
4,py_14_sid_a081
5,py_16_sid_a081
6,py_17_sid_a081
7,py_18_sid_a081
8,py_19_sid_a081
9,py_1_sid_a081


In [21]:
# Define the function
def simple_fun(x):
    return 2 * x + 5

# Apply the function to the 'Sepal.Length' column
calculated = simple_fun(iris_hex['sepal length (cm)'])

# Combine the original 'Sepal.Length' column with the calculated values
combined = iris_hex['sepal length (cm)'].cbind(calculated)

# Display the combined result
h2o.as_list(combined)

/usr/local/lib/python3.10/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install datatable (for Python 3.9 or lower), or polars and pyarrow (for Python 3.10 or above) and activate it using:

with h2o.utils.threading.local_context(polars_enabled=True, datatable_enabled=True):
    pandas_df = h2o_df.as_data_frame()

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


,sepal length (cm),sepal length (cm)0
0,5.1,15.2
1,4.9,14.8
2,4.7,14.4
3,4.6,14.2
4,5.0,15.0
...,...,...
145,6.7,18.4
146,6.3,17.6
147,6.5,18.0
148,6.2,17.4


# Running Models

## Classification (1)

In [22]:
from h2o.estimators import H2OGradientBoostingEstimator

In [ ]:
# Train a GBM model
iris_gbm = H2OGradientBoostingEstimator()
iris_gbm.train(y = 'target',
               x = iris_df.columns[:-1].tolist(),
               training_frame = iris_train)

In [24]:
# Predict on the test set
predictions = iris_gbm.predict(iris_test)
print(predictions)

gbm prediction progress: |███████████████████████████████████████████████████████| (done) 100%
  predict        p0           p1           p2
        0  0.99861   0.000700155  0.000690257
        0  0.99861   0.000700194  0.0006903
        0  0.991104  0.0084341    0.00046142
        0  0.998743  0.000772911  0.00048404
        0  0.997291  0.0023294    0.000379907
        0  0.998743  0.000772674  0.000483892
        0  0.99774   0.00186488   0.000395214
        0  0.998743  0.000772674  0.000483892
        0  0.998397  0.00115858   0.000444536
        0  0.998635  0.000729049  0.00063585
[33 rows x 4 columns]



In [25]:
# Generate the performance metrics for the test set
performance = iris_gbm.model_performance(iris_test)

In [26]:
performance.confusion_matrix()

0,1,2,Error,Rate
15.0,0.0,0.0,0.0,0 / 15
0.0,11.0,1.0,0.0833333,1 / 12
0.0,1.0,5.0,0.1666667,1 / 6
15.0,12.0,6.0,0.0606061,2 / 33


In [29]:
performance.hit_ratio_table()

k,hit_ratio
1,0.9393939
2,1.0
3,1.0


## Classification (2)

In [30]:
from h2o.estimators import H2OGeneralizedLinearEstimator

# Import the prostate dataset
prostate_hex = h2o.import_file('https://raw.github.com/h2oai/h2o/master/smalldata/logreg/prostate.csv',
                               destination_frame='prostate_hex')


Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


In [ ]:
# Train a GLM model for logistic classification
prostate_glm = H2OGeneralizedLinearEstimator(family = 'binomial', nfolds = 10)
prostate_glm.train(y='CAPSULE', x=['AGE', 'RACE', 'PSA', 'DCAPS'], training_frame = prostate_hex)

In [ ]:
# Get cross-validation metrics
prostate_glm.cross_validation_metrics_summary().as_data_frame()

In [ ]:
# Predict on the same dataset
prostate_glm.predict(prostate_hex)

In [34]:
# Calculate AUC
auc = prostate_glm.auc()
print(f'AUC: {auc}')

AUC: 0.7176009904696092


In [ ]:
# Generate the confusion matrix
confusion_matrix = prostate_glm.confusion_matrix()
print(confusion_matrix)

In [ ]:
# Get performance metrics on the training data
performance = prostate_glm.model_performance(prostate_hex)
print(performance)

## Regression

The Loblolly data frame has 84 rows and 3 columns of records of the growth of Loblolly pine trees.

Kung, F. H. (1986), Fitting logistic growth curve with predetermined carrying capacity, in Proceedings of the Statistical Computing Section, American Statistical Association, 340–343.

Pinheiro, J. C. and Bates, D. M. (2000) Mixed-effects Models in S and S-PLUS, Springer.

In [ ]:
# Import the Loblolly dataset from the CSV file
loblolly_hex = h2o.import_file('loblolly.csv', destination_frame = 'Loblolly_hex')
loblolly_hex

In [ ]:
# Train a GLM model for regression
loblolly_glm = H2OGeneralizedLinearEstimator(nfolds = 10)
loblolly_glm.train(y='height', x=['age'], training_frame = loblolly_hex)

In [ ]:
# Predict using the GLM model
glm_predict = loblolly_glm.predict(loblolly_hex)
print(glm_predict)

In [40]:
from h2o.automl import H2OAutoML

In [ ]:
# Train an AutoML model
loblolly_automl = H2OAutoML(max_runtime_secs = 60)
loblolly_automl.train(y='height', x=['age'], training_frame=loblolly_hex)

In [ ]:
loblolly_automl.get_leaderboard()

In [ ]:
# Get the leader model from AutoML
leader_model = loblolly_automl.leader
print(leader_model)

# Predict using the AutoML leader model
automl_predict = leader_model.predict(loblolly_hex)
print(automl_predict)

## Clustering

In [44]:
from h2o.estimators import H2OKMeansEstimator

In [ ]:
iris_kmeans = H2OKMeansEstimator(k = 3)
iris_kmeans.train(x = list(range(0, 4)),
                  training_frame = iris_hex)


In [ ]:
# Print model details
print(iris_kmeans)

In [ ]:
# Show cluster centers
print("Cluster Centers: ", iris_kmeans.centers())

In [ ]:
# Show clustering metrics
print("Clustering Metrics: ", iris_kmeans._model_json['output']['training_metrics'])

## PCA

In [49]:
from h2o.estimators import H2OPrincipalComponentAnalysisEstimator

In [ ]:
iris_pca_data = iris_hex[:, :-1]

# Train a PCA model
iris_pca = H2OPrincipalComponentAnalysisEstimator(k=2, transform='NORMALIZE')
iris_pca.train(training_frame=iris_pca_data)

In [ ]:
# Print model details
print(iris_pca)

In [ ]:
# Show the principal components
print("Principal Components: ", iris_pca._model_json['output']['importance'])

In [53]:
# Get the principal component projections
pca_projections = iris_pca.predict(iris_pca_data)
print(pca_projections)

pca prediction progress: |███████████████████████████████████████████████████████| (done) 100%
     PC1         PC2
0.630703   0.107578
0.622905  -0.10426
0.66952   -0.0514171
0.654153  -0.102885
0.648788   0.133488
0.535273   0.289616
0.656538   0.0107245
0.62578    0.0571335
0.675644  -0.200703
0.645645  -0.067208
[150 rows x 2 columns]



# Case Study (MNIST)

In [54]:
# URLs to the train and test datasets
train_file = 'https://h2o-public-test-data.s3.amazonaws.com/bigdata/laptop/mnist/train.csv.gz'
test_file = 'https://h2o-public-test-data.s3.amazonaws.com/bigdata/laptop/mnist/test.csv.gz'

# Import the datasets
train = h2o.import_file(train_file)
test = h2o.import_file(test_file)

# Display a brief summary of the data
train.summary()
test.summary()

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


<ipython-input-54-e73e24c90c96>:10: H2ODeprecationWarning: ``summary()`` is deprecated, please use ``show_summary()`` or ``get_summary()`` instead
  train.summary()


C1     C2     C3     C4     C5     C6     C7     C8     C9     C10    C11    C12    C13                 C14                   C15                C16                  C17    C18    C19    C20    C21    C22    C23    C24    C25    C26    C27    C28    C29    C30    C31    C32    C33                    C34                    C35                   C36                   C37                  C38                C39                  C40                  C41                  C42                  C43                  C44                C45                  C46                C47                  C48                  C49                   C50                   C51                   C52                   C53    C54    C55    C56    C57    C58    C59                    C60                  C61                 C62                   C63                 C64                C65                 C66                 C67                 C68                 C69                 C70                 C71                 C72                 C73                 C74                C75                 C76                 C77                 C78                 C79                 C80                  C81                  C82                   C83    C84    C85    C86    C87                    C88                 C89                   C90                  C91                  C92                C93                 C94                 C95                 C96                 C97                 C98                 C99                 C100                C101                C102               C103               C104               C105                C106                C107                C108                C109                 C110                  C111              C112    C113    C114                   C115                  C116                C117                 C118                C119                C120                C121                C122               C123                C124                C125               C126               C127               C128               C129               C130               C131               C132               C133                C134               C135                C136               C137                C138               C139               C140                  C141    C142    C143                  C144               C145                C146                C147                C148               C149               C150                C151                C152               C153                C154                C155                C156                C157                C158                C159              C160                C161                C162                C163               C164                C165                C166                C167                 C168                C169    C170                    C171               C172                 C173                C174               C175               C176               C177                C178               C179                C180                C181                C182                C183                C184                C185                C186                C187                C188                C189               C190               C191               C192                C193                C194                C195                C196                C197                   C198                  C199                 C200                C201               C202               C203                C204                C205              C206               C207                C208                C209                C210                C211                C212                C213                C214                C215                C216               C217                C218               C219                C220                C221               C222                C223               C224                 C225                  C226                 C227   

C1     C2     C3     C4     C5     C6     C7     C8     C9     C10    C11    C12    C13    C14    C15    C16    C17    C18    C19    C20    C21    C22    C23    C24    C25    C26    C27    C28    C29    C30    C31    C32    C33    C34     C35     C36     C37                 C38                 C39                 C40                C41                C42                C43                C44                C45                C46                 C47                C48                  C49                 C50    C51    C52    C53    C54    C55    C56    C57    C58    C59    C60    C61    C62                C63                C64                C65                C66                C67                 C68                 C69                 C70                 C71                 C72                 C73                C74                 C75                C76                 C77                C78                C79                C80                C81     C82    C83    C84    C85    C86    C87    C88    C89                 C90                 C91                 C92                C93                 C94                 C95                 C96                 C97                 C98                 C99                C100                C101               C102               C103               C104               C105               C106                C107                C108              C109                C110              C111    C112    C113    C114    C115    C116                  C117               C118               C119                C120               C121                C122               C123                C124                C125              C126                C127               C128               C129               C130               C131               C132                C133                C134               C135                C136               C137                C138               C139               C140    C141    C142                 C143                   C144               C145               C146                C147                C148                C149                C150               C151               C152               C153               C154                C155                C156                C157                C158                C159               C160               C161               C162                C163                C164               C165               C166                C167              C168    C169    C170    C171                 C172               C173               C174               C175                C176               C177               C178               C179               C180                C181                C182                C183                C184                C185                C186                C187                C188                C189               C190                C191                C192                C193               C194               C195              C196                C197    C198                C199                C200               C201                C202              C203                C204                C205               C206               C207                C208                C209                C210                C211                C212              C213                C214                C215                C216                C217               C218                C219                C220                C221                C222                C223               C224                C225    C226               C227               C228               C229               C230                C231              C232                C233               C234                C235                C236                C237                C238                C239                C240                C241                C242                C243                C244                C245                C246             C247                C248    

In [55]:
# Specify the response and predictor columns
y = 'C785'
x = list(set(train.names) - set([y]))

# Encode the response column as categorical for classification
train[y] = train[y].asfactor()
test[y] = test[y].asfactor()

In [ ]:
from h2o.estimators import H2ODeepLearningEstimator
# The example below illustrates the relative simplicity underlying most H2O Deep Learning model parameter
# configurations, as a result of the default settings. Rectified linear activation is popular
# with image processing and has performed well on the MNIST database previously and dropout has been
# known to enhance performance on this dataset as well, so we train our model accordingly.

# Train a Deep Learning model and validate on a test set.

# Train the deep learning model
model = H2ODeepLearningEstimator(
    distribution='multinomial',
    activation='RectifierWithDropout',
    hidden=[200, 200, 200], # 784-200-200-200-10
    input_dropout_ratio=0.2, # 784 * 200 + 200 + 200 * 200 + 200 + 200 * 200 +200 + 200 * 10 + 10 =  239 410 parameters
    l1=1e-5,
    epochs=10,
    variable_importances=True
)

model.train(x=x, y=y, training_frame=train, validation_frame=test)

In [57]:
# View the specified parameters
print("Model parameters: ", model.params)

Model parameters:  {'model_id': {'default': None, 'actual': {'__meta': {'schema_version': 3, 'schema_name': 'ModelKeyV3', 'schema_type': 'Key<Model>'}, 'name': 'DeepLearning_model_python_1717403639876_65', 'type': 'Key<Model>', 'URL': '/3/Models/DeepLearning_model_python_1717403639876_65'}, 'input': None}, 'training_frame': {'default': None, 'actual': {'__meta': {'schema_version': 3, 'schema_name': 'FrameKeyV3', 'schema_type': 'Key<Frame>'}, 'name': 'py_33_sid_a081', 'type': 'Key<Frame>', 'URL': '/3/Frames/py_33_sid_a081'}, 'input': {'__meta': {'schema_version': 3, 'schema_name': 'FrameKeyV3', 'schema_type': 'Key<Frame>'}, 'name': 'py_33_sid_a081', 'type': 'Key<Frame>', 'URL': '/3/Frames/py_33_sid_a081'}}, 'validation_frame': {'default': None, 'actual': {'__meta': {'schema_version': 3, 'schema_name': 'FrameKeyV3', 'schema_type': 'Key<Frame>'}, 'name': 'py_34_sid_a081', 'type': 'Key<Frame>', 'URL': '/3/Frames/py_34_sid_a081'}, 'input': {'__meta': {'schema_version': 3, 'schema_name': 'Fr

In [ ]:
# Display all performance metrics
print("Model performance: ", model)

In [59]:
# Training set metrics
train_perf = model.model_performance(train)
print("Training performance: ", train_perf)

Training performance:  ModelMetricsMultinomial: deeplearning
** Reported on test data. **

MSE: 0.024053370982466692
RMSE: 0.15509149229556948
LogLoss: 0.0911854682473896
Mean Per-Class Error: 0.028490943953264986
AUC table was not computed: it is either disabled (model parameter 'auc_type' was set to AUTO or NONE) or the domain size exceeds the limit (maximum is 50 domains).
AUCPR table was not computed: it is either disabled (model parameter 'auc_type' was set to AUTO or NONE) or the domain size exceeds the limit (maximum is 50 domains).

Confusion Matrix: Row labels: Actual class; Column labels: Predicted class
0     1     2     3     4     5     6     7     8     9     Error      Rate
----  ----  ----  ----  ----  ----  ----  ----  ----  ----  ---------  --------------
5844  1     19    6     3     8     21    2     13    6     0.0133378  79 / 5,923
1     6640  52    9     5     3     1     14    15    2     0.015129   102 / 6,742
13    7     5812  26    24    3     18    26    25 

In [60]:
# Validation set metrics
valid_perf = model.model_performance(test)
print("Validation performance: ", valid_perf)

Validation performance:  ModelMetricsMultinomial: deeplearning
** Reported on test data. **

MSE: 0.028610970731963618
RMSE: 0.1691477777919758
LogLoss: 0.11646824606917394
Mean Per-Class Error: 0.03300755699704586
AUC table was not computed: it is either disabled (model parameter 'auc_type' was set to AUTO or NONE) or the domain size exceeds the limit (maximum is 50 domains).
AUCPR table was not computed: it is either disabled (model parameter 'auc_type' was set to AUTO or NONE) or the domain size exceeds the limit (maximum is 50 domains).

Confusion Matrix: Row labels: Actual class; Column labels: Predicted class
0    1     2     3     4    5    6    7     8    9    Error       Rate
---  ----  ----  ----  ---  ---  ---  ----  ---  ---  ----------  ------------
968  0     1     2     1    2    3    1     2    0    0.0122449   12 / 980
0    1125  6     0     0    0    3    0     1    0    0.00881057  10 / 1,135
6    1     994   8     4    0    4    8     5    2    0.0368217   38 / 1,03

In [66]:
# Get MSE only for validation set
mse_valid = valid_perf.mse()
print("Validation MSE: ", mse_valid)

Validation MSE:  0.028610970731963618


In [62]:
# Variable importance
var_importance = model.varimp()
print("Variable importance: ", var_importance)

Variable importance:  [('C76', 1.0, 1.0, 0.0029891531397159402), ('C78', 0.9820566177368164, 0.9820566177368164, 0.0029355176222868214), ('C77', 0.9753942489624023, 0.9753942489624023, 0.0029156027817468363), ('C770', 0.9691189527511597, 0.9691189527511597, 0.0028968449603743525), ('C769', 0.9617758989334106, 0.9617758989334106, 0.002874895447999925), ('C768', 0.9557488560676575, 0.9557488560676575, 0.0028568796938945565), ('C773', 0.949530839920044, 0.949530839920044, 0.002838293091404113), ('C75', 0.9327151775360107, 0.9327151775360107, 0.002788028501392477), ('C771', 0.8955176472663879, 0.8955176472663879, 0.002676839386997355), ('C772', 0.8902101516723633, 0.8902101516723633, 0.0026609744698784478), ('C295', 0.885613203048706, 0.885613203048706, 0.0026472334864669302), ('C767', 0.8789734840393066, 0.8789734840393066, 0.002627386349543152), ('C323', 0.8760762214660645, 0.8760762214660645, 0.0026187259880257638), ('C774', 0.862788736820221, 0.862788736820221, 0.0025790076615777135), 

In [63]:
# Predictions
pred = model.predict(test)
print("Predictions: ", pred.head())

deeplearning prediction progress: |██████████████████████████████████████████████| (done) 100%
Predictions:    predict           p0           p1           p2           p3           p4           p5           p6           p7           p8           p9
        8  1.11273e-08  6.22812e-07  9.90168e-07  7.03138e-06  4.21413e-08  2.77153e-06  9.31764e-08  8.1077e-09   0.999987     1.64006e-06
        3  6.05483e-11  3.72786e-08  3.98553e-08  0.999984     4.27197e-10  1.06254e-06  2.81699e-12  1.19649e-06  6.02361e-06  7.65789e-06
        8  0.0812754    0.000508378  0.0128297    0.00772221   0.00496229   0.0985099    0.213177     0.000675486  0.574423     0.00591656
        0  0.997569     2.70662e-08  3.42211e-06  2.442e-07    1.13628e-06  5.87922e-06  0.00241297   2.83724e-07  4.99578e-06  2.52263e-06
        1  7.34495e-11  0.999999     3.47374e-08  1.37428e-08  9.21581e-08  2.21371e-09  1.08854e-08  2.22102e-07  3.51128e-07  1.77772e-08
        5  1.61587e-05  1.69459e-05  2.37344e-05  0.

In [64]:
from h2o.grid.grid_search import H2OGridSearch

# Define hyperparameters for grid search
hidden_opt = [[100, 300, 100], [500, 500, 500]]
l1_opt = [1e-5, 1e-7]
hyper_params = {'hidden': hidden_opt, 'l1': l1_opt}

# Define search criteria
search_criteria = {'strategy': 'RandomDiscrete', 'max_runtime_secs': 900}

# Perform grid search
model_grid = H2OGridSearch(
    model=H2ODeepLearningEstimator(
        distribution='multinomial',
        epochs=5,
        validation_frame=test
    ),
    hyper_params=hyper_params,
    search_criteria=search_criteria
)

model_grid.train(x=x, y=y, training_frame=train)

# Print out all prediction errors of the models
print(model_grid)

deeplearning Grid Build progress: |██████████████████████████████████████████████| (done) 100%
Hyper-Parameter Search Summary: ordered by increasing logloss
    hidden           l1     model_ids                                                               logloss
--  ---------------  -----  ----------------------------------------------------------------------  ---------
    [500, 500, 500]  1e-07  Grid_DeepLearning_py_33_sid_a081_model_python_1717403639876_66_model_1  0.168967


In [65]:
# Print out the Test MSE for all of the models
for model_id in model_grid.model_ids:
    model = h2o.get_model(model_id)
    mse = model.model_performance(valid=True).mse()
    print(f"Model ID: {model_id}, Test set MSE: {mse:.6f}")

Model ID: Grid_DeepLearning_py_33_sid_a081_model_python_1717403639876_66_model_1, Test set MSE: 0.030876


In [67]:
h2o.cluster().shutdown()

H2O session _sid_a081 closed.


<ipython-input-67-1edf85295eae>:1: H2ODeprecationWarning: Deprecated, use ``h2o.cluster().shutdown()``.
  h2o.shutdown()


# Exercise (Titanic): 45 minutes

[Kaggle](https://www.kaggle.com/competitions/titanic)



